In [1]:
import csv

# read demand data
f = open("module2_quiz_P2a.csv")
csvfile = csv.DictReader(f)
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

Groups = []
Demand = {}
Months = range(1,6)
for row in table:
    Groups.append(row['Demand (tonne)'])
    for m in Months:
        Demand[(row['Demand (tonne)'], m)] = float(row[str("Month "+str(m))])

# read cost data
f = open("module2_quiz_P2b.csv")
csvfile = csv.DictReader(f)
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()
                
Cost = {}
for row in table:
    for m in Months:
        Cost[(row['Production cost ($/tonne)'], m)] = float(row[str("Month "+str(m))])

In [2]:
Groups

['Large', 'Medium', 'Small']

In [3]:
Demand

{('Large', 1): 31.0,
 ('Large', 2): 25.0,
 ('Large', 3): 45.0,
 ('Large', 4): 59.0,
 ('Large', 5): 44.0,
 ('Medium', 1): 50.0,
 ('Medium', 2): 41.0,
 ('Medium', 3): 70.0,
 ('Medium', 4): 80.0,
 ('Medium', 5): 60.0,
 ('Small', 1): 42.0,
 ('Small', 2): 30.0,
 ('Small', 3): 55.0,
 ('Small', 4): 72.0,
 ('Small', 5): 56.0}

In [4]:
Months

range(1, 6)

In [5]:
Cost

{('Large', 1): 286.0,
 ('Large', 2): 273.0,
 ('Large', 3): 305.0,
 ('Large', 4): 337.0,
 ('Large', 5): 353.0,
 ('Medium', 1): 304.0,
 ('Medium', 2): 290.0,
 ('Medium', 3): 324.0,
 ('Medium', 4): 359.0,
 ('Medium', 5): 375.0,
 ('Small', 1): 313.0,
 ('Small', 2): 299.0,
 ('Small', 3): 334.0,
 ('Small', 4): 370.0,
 ('Small', 5): 386.0}

In [6]:
ProcessingCapacity = 175
HoldingCost = 50
StorageCapacity = 50
StorageRot = 0.04

In [7]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [8]:
# variables
Processed = mdl.continuous_var_matrix(Groups, Months, lb=0, name='Processed')

In [9]:
# auxiliary variables: inventory at end of month
Inventory = mdl.continuous_var_matrix(Groups, Months, lb=0,  name='Inventory')

In [10]:
# objective: minimize cost
TotalProcessingCost = mdl.sum(Processed[(g,m)] * Cost[(g,m)] for g in Groups for m in Months)
TotalHoldingCost = mdl.sum(HoldingCost*Inventory[(g,m)] for g in Groups for m in Months)
mdl.minimize(TotalProcessingCost + TotalHoldingCost)

mdl.add_kpi(TotalProcessingCost, 'Total processing cost')
mdl.add_kpi(TotalHoldingCost, 'Total holding cost')

DecisionKPI(name=Total holding cost,expr=50Inventory_Large_1+50Inventory_Large_2+50Inventory_Large_3+50In..)

In [11]:
# total processing per month constraint - at most 175 tonnes of potatoes/month
# total storage capacity per month constraint - 50 tonnes
for m in Months:
    mdl.add_constraint(mdl.sum(Processed[(g,m)] for g in Groups) <= ProcessingCapacity)
    mdl.add_constraint(mdl.sum(Inventory[(g,m)] for g in Groups) <= StorageCapacity)

In [12]:
for m in range(6):
    for g in Groups:
        if m == 0:
            if g == 'Medium':
                Inventory[(g, m)] = 5.3
            elif g == 'Large':
                Inventory[(g, m)] = 2.3
            else:
                Inventory[(g, m)] = 3.7
                

In [13]:
for m in range(6):
    for g in Groups:
        print(Inventory[(g,m)])


2.3
5.3
3.7
Inventory_Large_1
Inventory_Medium_1
Inventory_Small_1
Inventory_Large_2
Inventory_Medium_2
Inventory_Small_2
Inventory_Large_3
Inventory_Medium_3
Inventory_Small_3
Inventory_Large_4
Inventory_Medium_4
Inventory_Small_4
Inventory_Large_5
Inventory_Medium_5
Inventory_Small_5


In [14]:
for g in Groups:
    for m in Months:
        if m == 1:
            mdl.add_constraint(Inventory[(g, m)] == Inventory[(g,m-1)] + Processed[(g, m)] - Demand[(g, m)])
        else:
            mdl.add_constraint(Inventory[(g, m)] == Inventory[(g, m-1)]* (1-StorageRot) + Processed[(g, m)] - Demand[(g, m)])
            

In [15]:
# solve 
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00298882,status='optimal')

In [16]:
mdl.print_solution()

objective: 253176.004
status: OPTIMAL_SOLUTION(2)
  Processed_Large_1=28.700
  Processed_Large_2=25.000
  Processed_Large_3=45.000
  Processed_Large_4=59.000
  Processed_Large_5=44.000
  Processed_Medium_1=44.700
  Processed_Medium_2=41.000
  Processed_Medium_3=70.000
  Processed_Medium_4=80.000
  Processed_Medium_5=60.000
  Processed_Small_1=38.300
  Processed_Small_2=63.854
  Processed_Small_3=60.000
  Processed_Small_4=36.000
  Processed_Small_5=56.000
  Inventory_Small_2=33.854
  Inventory_Small_3=37.500


In [17]:
mdl.report()

* model docplex_model1 solved with objective = 253176.004
*  KPI: Total processing cost = 249608.296
*  KPI: Total holding cost    = 3567.708
